# 9.8 — An Idempotent `foreachBatch`

**Chapter 9, §9.7.2, and Exercise 7.** *A micro-batch is processed twice. Does the
sink notice?*

`foreachBatch` is the sink of last resort and the most useful one: Spark hands the
callback an ordinary static DataFrame and the batch's identifier, and everything
the batch API can do is available inside it. What it does not hand over is any
guarantee. If a batch fails partway, Spark reprocesses the whole batch and calls
the callback again with the same rows, and whether that is harmless depends
entirely on what the callback does with them.

The two callbacks below differ in one line. One appends; the other overwrites a
partition named by the batch identifier. Calling each of them twice with the same
batch is the whole experiment, and it is a fair imitation of a failure, because
reprocessing is exactly what Spark does.

In [1]:
import os, shutil, tempfile

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch09-foreachbatch")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

spark = (
    SparkSession.builder.appName("CS777-9.8")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

## Two micro-batches of readings

Four rows, in two batches. Small enough to count by eye, which is the point: every
number below is one you can check.

In [2]:
BATCHES = {
    0: [("boston", 12.1), ("denver", 3.8)],
    1: [("miami", 27.0), ("seattle", 9.4)],
}
COLUMNS = ["station", "temperature_c"]

def batch_frame(batch_id):
    return spark.createDataFrame(BATCHES[batch_id], COLUMNS)

for batch_id, rows in BATCHES.items():
    print(batch_id, rows)

0 [('boston', 12.1), ('denver', 3.8)]
1 [('miami', 27.0), ('seattle', 9.4)]


## The appending callback

It writes what it is given, which is the obvious thing to do and the wrong one.
Batch 0 is delivered twice, and the output now holds two copies of it.

In [3]:
NAIVE = os.path.join(SCRATCH, "naive")

def append_batch(frame, batch_id):
    frame.write.mode("append").parquet(NAIVE)

append_batch(batch_frame(0), 0)
append_batch(batch_frame(1), 1)
print("after both batches:      ", spark.read.parquet(NAIVE).count(), "rows")

append_batch(batch_frame(0), 0)     # the failure: batch 0 arrives again
print("after batch 0 is replayed:", spark.read.parquet(NAIVE).count(), "rows")
spark.read.parquet(NAIVE).orderBy("station").show(truncate=False)

after both batches:       4 rows
after batch 0 is replayed: 6 rows


+-------+-------------+
|station|temperature_c|
+-------+-------------+
|boston |12.1         |
|boston |12.1         |
|denver |3.8          |
|denver |3.8          |
|miami  |27.0         |
|seattle|9.4          |
+-------+-------------+



## The idempotent callback

The identifier Spark passes is the missing piece. Writing each batch into its own
partition, with `overwrite` confined to that partition, makes a replay a rewrite
of the same bytes rather than an addition.

`partitionOverwriteMode` set to `dynamic` is what confines it: without that,
`overwrite` would clear the whole directory and lose every other batch.

In [4]:
IDEMPOTENT = os.path.join(SCRATCH, "idempotent")
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

def upsert_batch(frame, batch_id):
    (
        frame.withColumn("batch_id", lit(batch_id))
        .write.mode("overwrite")
        .partitionBy("batch_id")
        .parquet(IDEMPOTENT)
    )

upsert_batch(batch_frame(0), 0)
upsert_batch(batch_frame(1), 1)
print("after both batches:      ", spark.read.parquet(IDEMPOTENT).count(), "rows")

upsert_batch(batch_frame(0), 0)     # the same failure, the same replay
print("after batch 0 is replayed:", spark.read.parquet(IDEMPOTENT).count(), "rows")
spark.read.parquet(IDEMPOTENT).orderBy("station").show(truncate=False)

after both batches:       4 rows
after batch 0 is replayed: 4 rows


+-------+-------------+--------+
|station|temperature_c|batch_id|
+-------+-------------+--------+
|boston |12.1         |0       |
|denver |3.8          |0       |
|miami  |27.0         |1       |
|seattle|9.4          |1       |
+-------+-------------+--------+



In [5]:
# The comparison, as an assertion that runs. A later edit that breaks the
# correspondence fails here rather than in someone's warehouse.
naive_rows = spark.read.parquet(NAIVE).count()
idempotent_rows = spark.read.parquet(IDEMPOTENT).count()
print("appending callback: ", naive_rows, "rows for 4 distinct readings")
print("idempotent callback:", idempotent_rows, "rows for 4 distinct readings")
assert naive_rows == 6 and idempotent_rows == 4

appending callback:  6 rows for 4 distinct readings
idempotent callback: 4 rows for 4 distinct readings


## Using it as a real sink

Nothing above needed a streaming query, because a replay is just the callback
being invoked twice. Wired into a query it looks like this, and the callback is
unchanged.

In [6]:
STREAM_OUT = os.path.join(SCRATCH, "streamed")
INPUT = os.path.join(SCRATCH, "input")
os.makedirs(INPUT, exist_ok=True)
for batch_id, rows in BATCHES.items():
    with open(os.path.join(INPUT, f"{batch_id}.csv"), "w") as fh:
        for station, temperature in rows:
            fh.write(f"{station},{temperature}\n")

def sink(frame, batch_id):
    (
        frame.withColumn("batch_id", lit(batch_id))
        .write.mode("overwrite")
        .partitionBy("batch_id")
        .parquet(STREAM_OUT)
    )

query = (
    spark.readStream.schema("station STRING, temperature_c DOUBLE")
    .option("maxFilesPerTrigger", 1)
    .csv(INPUT)
    .writeStream.foreachBatch(sink)
    .option("checkpointLocation", os.path.join(SCRATCH, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
print(spark.read.parquet(STREAM_OUT).count(), "rows written through foreachBatch")

4 rows written through foreachBatch


## What this shows

End-to-end exactly-once needs three things at once: an input that can be replayed,
progress that is checkpointed, and output that does not care how often it is
written. Remove any one and the guarantee is gone. The first two Spark provides;
the third is the callback's responsibility, and `foreachBatch` is where the course
hands it to you.

The batch identifier is the whole mechanism. It is stable across a retry, which
makes it a key you can overwrite against. A callback that appends has no such key
and no way to tell a first attempt from a second.

In [7]:
spark.stop()